# 00 — Exploration du jeu de données FoodSeg103**No GPU required.** CPU runtime is sufficient and starts faster.This notebook answers the question that governs every downstream stage: **which ten classesshould NutriVision detect?**The class map currently committed to `ml/class_map.yaml` contains placeholder category ids.Guessing them is the single highest-risk step in the pipeline, because a wrong id trains themodel on the wrong ingredient and produces labels that are structurally valid — so`validate_dataset.py` cannot detect the error. This notebook replaces the guess with ameasurement.**Output:** a `ml/class_map.yaml` chosen by instance count, plus the figures for the"Stratégie de Données" slide.

## 1 — Setup

In [ ]:
!pip install -q opencv-python-headless pyyaml pandas matplotlib tqdmimport os, subprocessREPO = "Zen-Daitsu/Nutrivision"if os.path.exists("/content/Nutrivision"):    !rm -rf /content/Nutrivisionsubprocess.run(["git","clone","--depth","1",f"https://github.com/{REPO}.git",                "/content/Nutrivision"], check=True)%cd /content/Nutrivision

In [ ]:
from google.colab import drivedrive.mount('/content/drive')SRC = "/content/drive/MyDrive/Nutrivision/FoodSeg103"   # adjust to your Drive layoutimport osassert os.path.isdir(SRC), f"Not found: {SRC}"os.makedirs("data", exist_ok=True)if not os.path.exists("data/FoodSeg103"):    os.symlink(SRC, "data/FoodSeg103")ROOT = "data/FoodSeg103"!ls {ROOT}

## 2 — Read the authoritative category listFoodSeg103 ships 104 categories, id 0 being background. Everything downstream depends onthese ids being read rather than assumed.

In [ ]:
import glob, oscandidates = glob.glob(f"{ROOT}/**/category_id.txt", recursive=True) + \             glob.glob(f"{ROOT}/**/*categor*", recursive=True)print("candidate files:", candidates[:10])CAT_FILE = candidates[0]categories = {}for line in open(CAT_FILE, encoding="utf-8"):    line = line.strip()    if not line:        continue    parts = line.split(maxsplit=1)          # "12  apple"  or  "12\tapple"    if len(parts) == 2 and parts[0].isdigit():        categories[int(parts[0])] = parts[1].strip()print(f"\n{len(categories)} categories parsed from {CAT_FILE}\n")for cid, name in sorted(categories.items()):    print(f"{cid:4d}  {name}")

## 3 — Measure instance countsScans every training mask and counts how many images contain each category, and how manypixels each occupies. This takes a few minutes; it is the empirical basis for class selection.A class present in only 40 images cannot be learned no matter how many epochs you run —`validate_dataset.py` enforces a floor of 50 instances for exactly this reason.

In [ ]:
import cv2, numpy as np, globfrom collections import Counterfrom tqdm.auto import tqdmMASK_DIR = f"{ROOT}/Images/ann_dir/train"masks = sorted(glob.glob(f"{MASK_DIR}/*.png"))print(f"{len(masks)} training masks")image_count = Counter()   # in how many images does this category appearpixel_count = Counter()   # total pixels across the corpusfor path in tqdm(masks, desc="scanning masks"):    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)    if m is None:        continue    ids, counts = np.unique(m, return_counts=True)    for cid, n in zip(ids, counts):        cid = int(cid)        if cid == 0:            continue        image_count[cid] += 1        pixel_count[cid] += int(n)print(f"\n{len(image_count)} distinct categories observed")

In [ ]:
import pandas as pddf = pd.DataFrame([    {"category_id": cid,     "name": categories.get(cid, f"<unknown {cid}>"),     "images": image_count[cid],     "megapixels": round(pixel_count[cid] / 1e6, 2),     "mean_px_per_image": int(pixel_count[cid] / image_count[cid])}    for cid in sorted(image_count)]).sort_values("images", ascending=False).reset_index(drop=True)pd.set_option("display.max_rows", 120)df

### Distribution — the long tail is the story

In [ ]:
import matplotlib.pyplot as pltfig, ax = plt.subplots(figsize=(14, 5))top = df.head(40)ax.bar(range(len(top)), top["images"], color="#7FD1B9")ax.axhline(50, color="#E0715F", ls="--", lw=1.5, label="viability floor (50 images)")ax.set_xticks(range(len(top)))ax.set_xticklabels(top["name"], rotation=75, ha="right", fontsize=8)ax.set_ylabel("images containing this category")ax.set_title("FoodSeg103 — 40 most frequent categories in the training split")ax.legend()plt.tight_layout(); plt.show()viable = df[df["images"] >= 50]print(f"{len(viable)} of {len(df)} categories clear the 50-image floor")print(f"{len(df) - len(viable)} categories are unlearnable at this corpus size")

## 4 — Select the ten classes, by measurementTwo criteria, in order:1. **Sufficient instances.** Below roughly 200 images a class trains poorly even when it   clears the contract floor.2. **Nutritional distinctness.** Ten classes that are all leafy greens teach the model   nothing useful about macros. The target set spans protein, starch, and vegetable.Adjust `WANTED` to match the actual category names printed in step 2 — spelling inFoodSeg103 will not match my guesses.

In [ ]:
# Left side: the substring to search for in FoodSeg103 category names.# Right side: the NutriVision class name.WANTED = {    "chicken":   "chicken_breast",    "beef":      "beef",    "egg":       "egg",    "rice":      "white_rice",    "broccoli":  "broccoli",    "spinach":   "spinach",    "tomato":    "tomato",    "avocado":   "avocado",    "potato":    "potato",    "carrot":    "carrot",}matches = []for needle, target in WANTED.items():    hits = df[df["name"].str.contains(needle, case=False, na=False)]    for _, row in hits.iterrows():        matches.append({"needle": needle, "target": target,                        "category_id": int(row["category_id"]),                        "foodseg_name": row["name"],                        "images": int(row["images"])})pd.DataFrame(matches).sort_values(["needle", "images"], ascending=[True, False])

Review the table above. Where a needle matched several categories, pick the one with themost images, and confirm the FoodSeg103 name actually means what you want. Then fill in`FINAL` below with the exact ids.

In [ ]:
# category_id -> NutriVision class id.  Fill in from the table above.FINAL = {    # 76: 0,   # chicken duck        -> chicken_breast    # 74: 1,   # beef steak          -> beef}assert FINAL, "Fill in FINAL from the table above before continuing."TARGETS = {}for cat_id, cls_id in FINAL.items():    TARGETS[cls_id] = WANTED[[k for k, v in WANTED.items()                              if v == list(WANTED.values())[cls_id]][0]] \                      if False else None# Simpler and explicit: name each class id directly.CLASS_NAMES = {    0: "chicken_breast", 1: "beef", 2: "egg", 3: "white_rice", 4: "broccoli",    5: "spinach", 6: "tomato", 7: "avocado", 8: "potato", 9: "carrot",}import yamlcfg = {"target_classes": {int(k): v for k, v in CLASS_NAMES.items() if k in FINAL.values()},       "source_to_target": {int(k): int(v) for k, v in FINAL.items()}}with open("ml/class_map.yaml", "w") as f:    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)print(open("ml/class_map.yaml").read())print("\nProjected instance counts for the selected classes:")for cat_id, cls_id in sorted(FINAL.items(), key=lambda x: x[1]):    n = image_count[cat_id]    flag = "OK" if n >= 200 else ("MARGINAL" if n >= 50 else "TOO FEW")    print(f"  {CLASS_NAMES[cls_id]:16s} cat {cat_id:3d}  {n:5d} images   {flag}")

## 5 — Verify visuallyOverlay the selected categories on real images. If a mask labelled `chicken_breast` issitting on a bowl of noodles, the id is wrong and no amount of training will fix it.

In [ ]:
import randomIMG_DIR = f"{ROOT}/Images/img_dir/train"PALETTE = [(255,180,84),(127,209,185),(224,113,95),(242,193,78),(150,180,255),           (200,120,220),(120,220,160),(255,140,140),(180,200,90),(90,200,220)]def show_category(cat_id, n=3):    hits = []    for path in masks:        m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)        if m is not None and (m == cat_id).sum() > 500:            hits.append(path)        if len(hits) >= n:            break    if not hits:        print(f"no example found for category {cat_id}"); return    fig, axes = plt.subplots(1, len(hits), figsize=(5*len(hits), 5))    axes = [axes] if len(hits) == 1 else axes    for ax, mp in zip(axes, hits):        stem = os.path.splitext(os.path.basename(mp))[0]        ip = next((p for p in glob.glob(f"{IMG_DIR}/{stem}.*")), None)        img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB)        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)        overlay = img.copy()        overlay[m == cat_id] = PALETTE[cat_id % len(PALETTE)]        ax.imshow(cv2.addWeighted(img, 0.55, overlay, 0.45, 0))        ax.axis("off")    plt.suptitle(f"category {cat_id} — {categories.get(cat_id, '?')}", fontsize=13)    plt.tight_layout(); plt.show()for cat_id in list(FINAL)[:4]:    show_category(cat_id)

## 6 — Commit the measured class mapCopy `ml/class_map.yaml` into Drive, then paste it into the repository through the GitHubweb editor. Commit message worth using:> `Replace guessed FoodSeg103 ids with measured category mapping`That diff is evidence of method. An examiner can see the ids were derived from the corpusrather than assumed.

In [ ]:
import shutil, osDEST = "/content/drive/MyDrive/Nutrivision/artifacts"os.makedirs(DEST, exist_ok=True)shutil.copy2("ml/class_map.yaml", DEST)df.to_csv(f"{DEST}/foodseg103_category_stats.csv", index=False)print("saved to", DEST)print("\nNext: 01_compile_and_train.ipynb")